# Lab 21: Calibrated scoring versus an LLM asked for a probability

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Fall-2026 | LLM mode: replay


In [2]:
# From the slide: Calibrating with scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import (
    CalibratedClassifierCV, calibration_curve)
from sklearn.metrics import brier_score_loss, roc_auc_score
log = pd.read_csv("data/reference/deal_log.csv")
FEATURES = ["revenue_usd_m", "growth", "ebitda_margin",
            "recurring_share", "top_customer_share",
            "asking_ev_ebitda"]
X_train, X_test, y_train, y_test, _, p_true_test = \
    train_test_split(log[FEATURES], log["advanced_to_ic"],
        log["p_true_advanced_to_ic"], test_size=0.3,
        random_state=7, stratify=log["advanced_to_ic"])
clf = CalibratedClassifierCV(
    estimator=LogisticRegression(max_iter=1000),
    method="isotonic", cv=5).fit(X_train, y_train)
p = clf.predict_proba(X_test)[:, 1]
print("AUC  ", roc_auc_score(y_test, p))
print("Brier", brier_score_loss(y_test, p))
print("Brier of the TRUE probs",
      brier_score_loss(y_test, p_true_test))  # floor
frac_pos, mean_pred = calibration_curve(
    y_test, p, n_bins=10)                   # the diagram

AUC   0.681842672413793
Brier 0.20523255553773348
Brier of the TRUE probs 0.18106188622222225


In [3]:
import numpy as np, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss, roc_auc_score
log = sd.deal_log(); ids = list(X_test.index)        # the identical test set for both approaches
rows = [{"deal_id": log.loc[i, "deal_id"], **{c: float(X_test.loc[i, c]) for c in X_test.columns}} for i in ids]
llm_scores = {s.deal_id: s.p for s in llm.structured(schemas.Scores, prompts.scorer_messages(rows), llm.P["llm"]["models"]["large"]).scores}
p_llm = np.array([llm_scores[log.loc[i, "deal_id"]] for i in ids]); p_mod = clf.predict_proba(X_test.loc[ids])[:, 1]; yt = y_test.loc[ids]
for name, p in (("calibrated model", p_mod), ("LLM-as-scorer", p_llm)):
    print(f"{name:16s} Brier {brier_score_loss(yt, p):.3f}  AUC {roc_auc_score(yt, p):.3f}  distinct values {len(set(np.round(p, 3)))}")
Path("reports/plots").mkdir(parents=True, exist_ok=True)
plt.plot(mean_pred, frac_pos, "o-", label="calibrated model"); plt.plot([0, 1], [0, 1], "--", label="perfect")
plt.xlabel("predicted"); plt.ylabel("observed"); plt.legend(); plt.savefig("reports/plots/reliability.png", dpi=120); print("saved reports/plots/reliability.png")

calibrated model Brier 0.205  AUC 0.682  distinct values 56
LLM-as-scorer    Brier 0.211  AUC 0.721  distinct values 11
saved reports/plots/reliability.png


Brier score rewards **calibration** (a stated 0.3 should come true about 30% of the time); AUC rewards only **ranking**. An approach can rank deals reasonably and still state badly calibrated probabilities.

**Exercise 21.** The LLM's probabilities cluster on round numbers. What does that do to its reliability diagram? Which approach should supply *probabilities* to the agent in Case Study 3, and would your answer change if the agent only needed a ranking?

In [4]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(6, 5))

for name, probabilities in [
    ("Calibrated model", p_mod),
    ("LLM", p_llm),
]:
    observed, predicted = calibration_curve(
        yt, probabilities, n_bins=10, strategy="uniform"
    )
    ax.plot(predicted, observed, "o-", label=name)

ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
ax.set_xlabel("Average predicted probability")
ax.set_ylabel("Fraction that actually advanced")
ax.set_title("Predicted probabilities versus observed outcomes")
ax.legend()
ax.grid(alpha=0.3)

# Display explicitly because this notebook uses the Agg backend.
from IPython.display import display
display(fig)
plt.close(fig)

<Figure size 600x500 with 1 Axes>

In [5]:
%matplotlib inline